In [2]:
import requests
from collections import defaultdict
from tqdm import tqdm
import time

In [ ]:
import requests
from collections import defaultdict
from tqdm import tqdm
import time

S = requests.Session()
S.headers.update({"User-Agent": "PoliticiansNetwork/1.0 (your@email.com)"})
BASE_PAGEVIEWS = "https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia/all-access"
BASE_WIKI = "https://en.wikipedia.org/w/api.php"

def get_top_pages_2024(n_days):
    """Fetch top 1000 articles per day for n_days in 2024"""
    from datetime import datetime, timedelta
    
    page_views = defaultdict(int)
    start = datetime(2024, 1, 1)
    days = [(start + timedelta(days=i)).strftime("%Y/%m/%d") for i in range(n_days)]
    
    for date in tqdm(days):
        url = f"{BASE_PAGEVIEWS}/{date}"
        try:
            r = S.get(url)
            articles = r.json()["items"][0]["articles"]
            for article in articles:
                title = article["article"]
                if any(title.startswith(skip) for skip in 
                       ["Special:", "Main_Page", "Wikipedia:", "Help:", "File:"]):
                    continue
                page_views[title] += article["views"]
        except Exception as e:
            print(f"Error on {date}: {e}")
        time.sleep(0.15)
    
    return page_views

page_views = get_top_pages_2024(n_days=360)
sorted_pages = sorted(page_views.items(), key=lambda x: x[1], reverse=True)
top_pages = [title for title, _ in sorted_pages[:5000]]  # grab top 5000 to filter from
print(f"Total pages collected: {len(top_pages)}")
print(top_pages[:20])

  0%|          | 0/360 [00:00<?, ?it/s]

  2%|▏         | 8/360 [00:03<02:26,  2.40it/s]

Error on 2024/01/08: 'items'


  9%|▉         | 34/360 [00:15<02:21,  2.31it/s]

Error on 2024/02/03: 'items'


...

Error on 2024/11/14: 'items'


 98%|█████████▊| 351/360 [03:14<00:04,  2.19it/s]

Error on 2024/12/16: 'items'


100%|██████████| 360/360 [03:19<00:00,  1.81it/s]

Total pages collected: 5000
['Cleopatra', 'Deaths_in_2024', 'YouTube', 'XXXTentacion', 'Pornhub', '2024_United_States_presidential_election', 'Kamala_Harris', 'Donald_Trump', '.xxx', 'Indian_Premier_League', 'Deadpool_&_Wolverine', 'Portal:Current_events', 'Project_2025', 'ChatGPT', 'Taylor_Swift', 'Elon_Musk', '2024_Indian_general_election', '2020_United_States_presidential_election', 'United_States', '2024_Summer_Olympics']


In [ ]:
def get_categories(title):
    """Get categories for a Wikipedia page"""
    params = {
        "action": "query",
        "titles": title,
        "prop": "categories",
        "cllimit": "max",
        "format": "json"
    }
    r = S.get(BASE_WIKI, params=params)
    page = next(iter(r.json()["query"]["pages"].values()))
    return [c["title"].lower() for c in page.get("categories", [])]


POLITICAL_KEYWORDS = [
    "president", "prime minister", "chancellor", "dictator", "king", "queen",
    "emperor", "pharaoh", "politician", "statesman", "head of state", "head of government", "senator", "minister",
    "secretary", "governor", "mayor", "monarch", "sultan"
]

def is_politician(title):
    categories = get_categories(title)
    return any(
        any(kw in category for kw in POLITICAL_KEYWORDS)
        for category in categories
    )

politicians = []

for title in tqdm(top_pages[:10000]):  # check top 10000 to find ~1000 politicians
    if is_politician(title):
        politicians.append(title)
    if len(politicians) >= 1000:
        break
    time.sleep(0.1)

print(f"\nFound {len(politicians)} politicians")


 67%|██████▋   | 3337/5000 [14:48<07:22,  3.75it/s]


Found 1000 politicians
['Cleopatra', '2024_United_States_presidential_election', 'Kamala_Harris', 'Donald_Trump', 'Project_2025', 'Elon_Musk', '2020_United_States_presidential_election', 'Facebook', 'Joe_Biden', 'Cristiano_Ronaldo', 'Ansel_Adams', 'Robert_F._Kennedy_Jr.', 'Sean_Combs', 'Tim_Walz', 'World_War_II', 'J._Robert_Oppenheimer', 'Chappell_Roan', 'House_of_the_Dragon', 'Lionel_Messi', 'Oppenheimer_(film)']


In [6]:
print(politicians[:50])

['Cleopatra', '2024_United_States_presidential_election', 'Kamala_Harris', 'Donald_Trump', 'Project_2025', 'Elon_Musk', '2020_United_States_presidential_election', 'Facebook', 'Joe_Biden', 'Cristiano_Ronaldo', 'Ansel_Adams', 'Robert_F._Kennedy_Jr.', 'Sean_Combs', 'Tim_Walz', 'World_War_II', 'J._Robert_Oppenheimer', 'Chappell_Roan', 'House_of_the_Dragon', 'Lionel_Messi', 'Oppenheimer_(film)', 'Simone_Biles', 'JD_Vance', '2016_United_States_presidential_election', 'Jimmy_Carter', 'Napoleon', 'Ryan_Reynolds', 'Barack_Obama', 'Wicked_(2024_film)', 'Adolf_Hitler', 'Elizabeth_II', 'Bob_Marley', 'Ariana_Grande', 'United_Kingdom', 'Civil_War_(film)', 'Gladiator_II', 'Charles_III', 'Zendaya', 'John_F._Kennedy', 'Elvis_Presley', 'Israel', 'BBC_World_Service', 'Lady_Gaga', 'Keir_Starmer', 'Nikki_Haley', 'The_Beekeeper_(2024_film)', 'Donald_J._Harris', 'The_Bear_(TV_series)', 'Jeffrey_Epstein', 'Russia', 'Baby_Reindeer']
